# 01 · Customer data understanding
**Company:** *Jamii Mobile*, a **fictional** Kenyan mobile operator. All subscriber data is **synthetic** (`src/generate_data.py`).
It does not describe Safaricom, Airtel, Telkom or any real customer.

Why synthetic? The usual public option, the IBM Telco Churn dataset, is a single-quarter snapshot of a fictional US company. It has no monthly history, so cohorts, monthly active users and declining activity can't be measured from it, and it has no usage, complaint or campaign data.

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format); pd.set_option("display.max_columns", 30)
engine = get_engine()
o = an.sql_output

In [2]:
counts = query(engine, """
SELECT 'customers' t, COUNT(*) n, 'one subscriber (SIM/account)' grain FROM telco.customers
UNION ALL SELECT 'subscriptions', COUNT(*), 'one plan period; open row = current plan' FROM telco.subscriptions
UNION ALL SELECT 'transactions', COUNT(*), 'one purchase: bundle, mobile-money fee or postpaid bill' FROM telco.transactions
UNION ALL SELECT 'reversals', COUNT(*), 'one refunded purchase' FROM telco.reversals
UNION ALL SELECT 'usage_monthly', COUNT(*), 'one customer x calendar month of usage' FROM telco.usage_monthly
UNION ALL SELECT 'complaints', COUNT(*), 'one complaint ticket' FROM telco.complaints
UNION ALL SELECT 'campaigns', COUNT(*), 'one marketing contact' FROM telco.campaigns
UNION ALL SELECT 'mv_customer_month', COUNT(*), 'derived: one row per subscribed customer per month' FROM telco.mv_customer_month""")
counts

,t,n,grain
0,reversals,3533,one refunded purchase
1,complaints,10559,one complaint ticket
2,customers,28496,one subscriber (SIM/account)
3,subscriptions,31539,one plan period; open row = current plan
4,campaigns,35523,one marketing contact
5,usage_monthly,560429,one customer x calendar month of usage
6,mv_customer_month,602836,derived: one row per subscribed customer per m...
7,transactions,1767025,"one purchase: bundle, mobile-money fee or post..."


## Why a customer-month grid?
Retention questions are about months in which **nothing happened**. Transaction and usage tables only hold rows when something *did* happen. `telco.mv_customer_month` creates a row for every month a line was open, from acquisition (or Jan 2024) to closure. A silent month is therefore a row with zero activity, not a missing row. Churn rates, ARPU, cohorts and the engagement trend are all computed from this grid.

In [3]:
query(engine, 'SELECT * FROM telco.mv_customer_month ORDER BY customer_id, month LIMIT 8')

,customer_id,month,cohort_month,customer_type,age_group,county,plan_type,tenure_months,revenue,transactions,data_mb,voice_minutes,sms_count,is_active,in_opening_base,is_churn_month
0,C000001,2024-01-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,38,616.00,3,"2,979.50",51.00,14,True,True,False
1,C000001,2024-02-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,39,786.00,7,"6,457.60",31.20,22,True,True,False
2,C000001,2024-03-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,40,868.00,4,"3,191.60",23.60,34,True,True,False
3,C000001,2024-04-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,41,903.00,2,"4,251.00",45.60,53,True,True,False
4,C000001,2024-05-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,42,663.00,3,"5,069.10",57.00,19,True,True,False
5,C000001,2024-06-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,43,660.00,3,"5,421.10",39.90,19,True,True,False
6,C000001,2024-07-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,44,396.00,4,"8,825.60",28.40,25,True,True,False
7,C000001,2024-08-01,2020-11-01,Prepaid,25-34,Kiambu,Prepaid Data Plus,45,305.00,1,"3,442.60",27.30,15,True,True,False


In [4]:
query(engine, """SELECT customer_type, COUNT(*) customers, MIN(acquisition_date) first_acq, MAX(acquisition_date) last_acq,
  ROUND(100.0*AVG((acquisition_date < DATE '2024-01-01')::int),1) AS pct_acquired_before_window FROM telco.customers GROUP BY 1""")

,customer_type,customers,first_acq,last_acq,pct_acquired_before_window
0,Prepaid,25093,2019-01-03,2026-10-31,45.70
1,Postpaid,3403,2019-01-03,2026-10-31,45.00


## Key definitions (from `sql/03_views.sql`)
| Term | Definition |
|---|---|
| Active in month | ≥1 purchase **or** any data/voice/SMS usage |
| Churned | line closed by the operator (`subscription_status = 'Churned'`), which happens **90 days after the last activity**, so churn is recorded about 3–4 months after the customer stopped |
| Monthly churn rate | lines closed in month *m* ÷ lines open on the 1st of *m* |
| ARPU | revenue in *m* ÷ active customers in *m* |
| Cohort retention | active in month *c+k* ÷ cohort size (fixed denominator); unobserved cells are blank |

Because of the 90-day closure rule, **no lines close in Jan–Apr 2024**: those customers were still inside their 90 days. Churn rates are therefore summarised from May 2024.